# Bootstrap Current Self-Consistency

This tutorial demonstrates how to optimize a quasi-symmetric equilibrium to have a self-consistent bootstrap current profile. 
This is performed by minimizing the difference between the toroidal currents $\langle J \cdot B \rangle$ computed from the MHD equilibrium and from the Redl formula. 
The Redl formula is only valid in the limit of perfect quasi-symmetry, so this procedure will not work for other configurations that are not quasi-symmetric. 

There are two methods that can be used, and both will be shown: 

1. Optimize the current profile for self-consistency
2. Iteratively solve the equilibrium with new current profiles

These methods should be equivalent, although one might be faster than the other depending on the particular problem. 

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("."))
sys.path.append(os.path.abspath("../../../"))

If you have access to a GPU, uncomment the following two lines before any DESC or JAX related imports. You should see about an order of magnitude speed improvement with only these two lines of code!

In [ ]:
# from desc import set_device
# set_device("gpu")

As mentioned in [DESC Documentation on performance tips](https://desc-docs.readthedocs.io/en/latest/performance_tips.html), one can use compilation cache directory to reduce the compilation overhead time. Note: One needs to create `jax-caches` folder manually.

In [ ]:
# import jax

# jax.config.update("jax_compilation_cache_dir", "../jax-caches")
# jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
# jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(linewidth=np.inf)
plt.rcParams["font.size"] = 14

from desc.compat import rescale
from desc.equilibrium import EquilibriaFamily
from desc.examples import get
from desc.grid import LinearGridFlux
from desc.objectives import (
    BootstrapRedlConsistency,
    FixAtomicNumber,
    FixBoundaryR,
    FixBoundaryZ,
    FixCurrent,
    FixElectronDensity,
    FixElectronTemperature,
    FixIonTemperature,
    FixPsi,
    ForceBalance,
    ObjectiveFunction,
)
from desc.plotting import plot_1d
from desc.profiles import PowerSeriesProfile, SplineProfile

As an example, we will reproduce the QA results from [Landreman et al. (2022)](https://doi.org/10.1063/5.0098166). 

We will start with the "precise QA" example equilibrium, scaled to the ARIES-CS reactor size. 

In [ ]:
eq0 = get("precise_QA")
eq0 = rescale(eq0, L=("R0", 10), B=("B0", 5.86))

This equilibrium has the vacuum profiles $p = 0 ~\text{Pa}$ and $\frac{2\pi}{\mu_0} I = 0 ~\text{A}$. 
Calculating the bootstrap current requires knowledge of the temperature and density profiles for each species in the plasma. 
We replace the pressure profile with the following kinetic profiles corresponding to $\langle\beta\rangle=2.5\%$: 

$n_e = n_i = 2.38\times10^{20} (1 - \rho^{10}) ~\text{m}^{-3}$

$T_e = T_i = 9.45\times10^{3} (1 - \rho^{2}) ~\text{eV}$

The temperature profiles must be given for both ions and electrons, but only the electron density profile is specified. 
The ion density profile is given by the effective atomic number $Z_{eff}$ as $n_i = n_e / Z_{eff}$. 
The plasma pressure will then be computed as 

$p = e (n_e T_e + n_i T_i)$.

In [ ]:
eq0.pressure = None  # must remove the pressure profile before setting kinetic profiles
eq0.atomic_number = PowerSeriesProfile([1])
eq0.electron_density = PowerSeriesProfile(params=[1, -1], modes=[0, 10]) * 2.38e20
eq0.electron_temperature = PowerSeriesProfile(params=[1, -1], modes=[0, 2]) * 9.45e3
eq0.ion_temperature = PowerSeriesProfile(params=[1, -1], modes=[0, 2]) * 9.45e3
eq0.current = PowerSeriesProfile([0])

We need to re-solve the equilibrium force balance with the new profiles. 

In [ ]:
eq0, _ = eq0.solve(objective="force", optimizer="lsq-exact", verbose=3)

Now we have our initial equilibrium, which does not have a self-consistent bootstrap current: 

In [ ]:
# the Redl formula is undefined at rho=0.0 and where the profiles vanish, so
# in our case here it is NaN at the axis and at rho=1.0
fig, ax = plot_1d(
    eq0,
    "<J*B> Redl",
    color="b",
    lw=2,
    label="Redl",
)
fig, ax = plot_1d(eq0, "<J*B>", color="r", lw=2, label="MHD", ax=ax)
ax.legend(loc="best")
ax.set_title("Initial Equilibrium");

We need to create a grid on which to evaluate the boootstrap current self-consistency. 
The bootstrap current is a radial profile, but the grid must have finite poloidal and toroidal resolution to accurately compute flux surface quantities. 
The Redl formula is undefined where the kinetic profiles vanish, so in our example we do not include points at $\rho=0$ or $\rho=1$. 

In [ ]:
grid = LinearGridFlux(
    M=eq0.M_grid,
    N=eq0.N_grid,
    NFP=eq0.NFP,
    sym=eq0.sym,
    rho=np.linspace(1 / eq0.L_grid, 1, eq0.L_grid) - 1 / (2 * eq0.L_grid),
)

Our current profile will be represented as a power series of the form: 

$I = c_0 + c_1 \rho + c_2 \rho^2 + \mathcal{O}(\rho^3)$

Physically, the current should vanish on the magnetic axis so $c_0 = 0$. 
And in order for the MHD equilibrium to be analytic, it should scale as $\mathcal{O}(\rho^2)$ near the magnetic axis so $c_1 = 0$ also. 
However, the Redl bootstrap current formula scales as $\mathcal{O}(\sqrt{\rho})$ near the magnetic axis. 
This is incorrect, because the drift-kinetic equation from the Redl formula does not account for finite orbit width effects that become important near the axis. 

Typically, we use even power series with `sym=True` for all equilibrium profiles to give the desired analycity conditions. 
For bootstrap current optimizations, it is recommended to use the full power series with `sym=False` while also enforcing $c_0 = c_1 = 0$. 
This prevents getting good self-consistency near the magnetic axis, but allows for good agreement throughout the rest of the plasma volume and results in high quality equilibria overall. 


In [ ]:
eq0.current = PowerSeriesProfile(np.zeros((eq0.L + 1,)), sym=False)

### 1. Optimization

In this method, we will optimize the current profile to minimize the self-consistency errors evaluated by the `BootstrapRedlConsistency` objective. 
This objective requires the helicity, which for QA is $(M, N) = (1, 0)$. 

In this example we will only optimize the current profile, so all other profiles and the plasma boundary are constrained to be fixed. 
It is recommended to use a very small value for `gtol` when optimizing the bootstrap current. 

In [ ]:
eq1 = eq0.copy()

In [ ]:
objective = ObjectiveFunction(
    BootstrapRedlConsistency(eq=eq1, grid=grid, helicity=(1, 0)),
)
constraints = (
    FixAtomicNumber(eq=eq1),
    FixBoundaryR(eq=eq1),
    FixBoundaryZ(eq=eq1),
    FixCurrent(eq=eq1, indices=[0, 1]),  # fix c_0=c_1=0 current profile coefficients
    FixElectronDensity(eq=eq1),
    FixElectronTemperature(eq=eq1),
    FixIonTemperature(eq=eq1),
    FixPsi(eq=eq1),
    ForceBalance(eq=eq1),
)
eq1, _ = eq1.optimize(
    objective=objective,
    constraints=constraints,
    optimizer="proximal-lsq-exact",
    maxiter=10,
    verbose=3,
)

When plotting the bootstrap current profiles, we see the MHD equilibrium now has very good agreement with the Redl formula. 

In [ ]:
fig, ax = plot_1d(eq1, "<J*B> Redl", color="b", lw=2, label="Redl")
fig, ax = plot_1d(eq1, "<J*B>", color="r", lw=2, label="MHD", ax=ax)
ax.legend(loc="best")
ax.set_title("Method 1: Optimization");

### 2. Iterative Solves

In this method, we iteratively solve the equilibrium with updated guesses for the current profile. 
The current profile is computed such that the parallel current is consistent with the Redl formula, according to Equation C3 in [Landreman & Catto (2012)](https://doi.org/10.1063/1.3693187). 
This is the same approach as STELLOPT VBOOT with SFINCS, and it usually converges in only a few iterations. 

In [ ]:
eq2 = eq0.copy()
fam2 = EquilibriaFamily(eq2)

In [ ]:
niters = 3
for k in range(niters):
    eq2 = eq2.copy()
    # compute new guess for the current profile, consistent with Redl formula
    data = eq2.compute("current Redl", grid)
    current = grid.compress(data["current Redl"])
    rho = grid.compress(data["rho"])
    # fit the current profile to a power series, with c_0=c_1=0
    XX = np.fliplr(np.vander(rho, eq2.L + 1)[:, :-2])
    eq2.c_l = np.pad(np.linalg.lstsq(XX, current, rcond=None)[0], (2, 0))
    # re-solve the equilibrium
    eq2, _ = eq2.solve(objective="force", optimizer="lsq-exact", verbose=3)
    fam2.append(eq2)

We can plot the current profile at each iteration to visualize how it changed: 

In [ ]:
fig, ax = plot_1d(fam2[0], "current", color="k", lw=2, label="0")
fig, ax = plot_1d(fam2[1], "current", color="g", lw=2, label="1", ax=ax)
fig, ax = plot_1d(fam2[2], "current", color="b", lw=2, label="2", ax=ax)
fig, ax = plot_1d(fam2[3], "current", color="r", lw=2, label="3", ax=ax)
ax.legend(loc="best");

With this method the MHD equilibrium also has very good agreement with the Redl formula. 

In [ ]:
fig, ax = plot_1d(eq2, "<J*B> Redl", color="b", lw=2, label="Redl")
fig, ax = plot_1d(eq2, "<J*B>", color="r", lw=2, label="MHD", ax=ax)
ax.legend(loc="best")
ax.set_title("Method 2: Iterative Solves");

### Comparison

Even though both methods give good self-consistency for the bootstrap current, they do result in slightly different coefficients for the current profile: 

In [ ]:
print(eq1.c_l)
print(eq2.c_l)

In this example, the first method of optimization gave better self-consistency but was noticeably slower than the second method of iterative solves. 

## Using Spline Profiles

We can also use splines to describe the current profile. Due to the many DOFs of splines and their local nature, they can be difficult to use with the first method shown above. So, we recommend using the iterative solve method if a spline current profile is desired. 

The code for the iterative method with splines is slightly different, as now we do not need to re-fit the current profile with a power series in order to update our spline. Instead, we choose our spline profile knots carefully to include a knot at the axis which we will keep at 0 (in order to enforce zero on-axis net toroidal current), and choosing the rest to align with the grid we will be computing the Redl bootstrap current at.

Remember that the Redl formula is not defined on-axis or on the boundary, so we must ensure we do not include a knot at $\rho=1$, as we have no way of updating that value.

In [ ]:
# Choose a denser grid for the spline profile so we can represent the bootstrap current profile well
L_grid = 25

grid = LinearGridFlux(
    M=eq0.M_grid,
    N=eq0.N_grid,
    NFP=eq0.NFP,
    sym=eq0.sym,
    rho=np.linspace(1 / L_grid, 1, L_grid) - 1 / (2 * L_grid),
)

eq2_spline = eq0.copy()
fam2 = EquilibriaFamily(eq2_spline)
# choose the rho values for the knots
rho = np.concatenate([np.array([0.0]), grid.nodes[grid.unique_rho_idx, 0]])
# make a SplineProfile for current, initially zero.
eq2_spline.current = SplineProfile(values=np.zeros_like(rho), knots=rho)

In [ ]:
niters = 3
for k in range(niters):
    eq2_spline = eq2_spline.copy()
    # compute new guess for the current profile, consistent with Redl formula
    data = eq2_spline.compute("current Redl", grid)
    current = grid.compress(data["current Redl"])
    rho = grid.compress(data["rho"])
    # add the rho=0 value which we enforce as zero current
    c_at_knots = np.concatenate([np.array([0.0]), current])
    eq2_spline.current.params = c_at_knots
    # re-solve the equilibrium
    eq2_spline, _ = eq2_spline.solve(
        objective="force", optimizer="lsq-exact", verbose=3
    )
    fam2.append(eq2_spline)

We can plot the current profile at each iteration to visualize how it changed: 

In [ ]:
fig, ax = plot_1d(fam2[0], "current", color="k", lw=2, label="0")
fig, ax = plot_1d(fam2[1], "current", color="g", lw=2, label="1", ax=ax)
fig, ax = plot_1d(fam2[2], "current", color="b", lw=2, label="2", ax=ax)
ax.legend(loc="best");

With this method the MHD equilibrium also has very good agreement with the Redl formula, now with a ``SplineProfile``.

In [ ]:
fig, ax = plot_1d(fam2[-1], "<J*B> Redl", color="b", lw=2, label="Redl")
fig, ax = plot_1d(fam2[-1], "<J*B>", color="r", lw=2, label="MHD", ax=ax)
ax.legend(loc="best")
ax.set_title("Method 2: Iterative Solves (splines)");